# Pull NLCD Data
This notebook extracts NLCD land cover rasters for each city in `data/city_coords.csv`, using the bounding box coordinates with an additional 0.05 degree margin on each side.

In [1]:
import pandas as pd
import ee

In [ ]:
ee.Authenticate()
ee.Initialize(project='ee-zcalhounnc')

## Load city coordinates

In [4]:
cities = pd.read_csv('../data/city_coords.csv')
cities

,Unnamed: 0,city,left,bottom,right,top
0,0,brooklyn,-74.041782,40.606855,-73.873780,40.762032
1,1,columbus,-83.183090,39.829610,-82.793823,40.150617
2,2,durham,-78.970188,35.933800,-78.813953,36.064848
3,3,omaha,-96.062930,41.176357,-95.900800,41.320967
4,4,little_rock,-92.509925,34.622287,-92.171693,34.816518
5,5,knoxville,-84.092528,35.904535,-83.829237,36.051725
6,6,cedar_rapids,-91.774582,41.865305,-91.601383,42.068685
7,7,lynchburg,-79.270337,37.333825,-79.124195,37.453620
8,8,richmond,-77.602524,37.441671,-77.328347,37.663227
9,9,san_francisco,-122.513720,37.705098,-122.365423,37.830112


## Helper functions

In [5]:
MARGIN = 0.2  # degrees

def coords_to_ee_geometry(left: float, bottom: float, right: float, top: float, margin: float = MARGIN) -> ee.Geometry:
    """Build an ee.Geometry rectangle from bounding box coords with an added margin."""
    return ee.Geometry.Rectangle([
        left  - margin,
        bottom - margin,
        right  + margin,
        top    + margin
    ])

In [6]:
def export_nlcd(city_name: str, aoi: ee.Geometry, year: int = 2021, export_to_drive: bool = True, output_scale_m: int = 30):
    """
    Export the NLCD land cover layer for a given AOI to Google Drive.
    Uses USGS/NLCD_RELEASES/2021_REL/NLCD (30 m resolution, EPSG:4326 export).
    """
    print(f"\n── Processing: {city_name}")

    nlcd = (
        ee.ImageCollection('USGS/NLCD_RELEASES/2021_REL/NLCD')
          .filter(ee.Filter.eq('system:index', str(year)))
          .first()
          .select('landcover')
          .clip(aoi)
    )

    if export_to_drive:
        task = ee.batch.Export.image.toDrive(
            image          = nlcd,
            description    = f'{city_name}',
            folder         = 'greenspace/nlcd_extra',
            fileNamePrefix = f'{city_name}',
            region         = aoi,
            scale          = output_scale_m,
            crs            = 'EPSG:4326',
            maxPixels      = 1e13,
            fileFormat     = 'GeoTIFF'
        )
        task.start()
        print(f"   Export task started → Drive/greenspace/{city_name}_NLCD.tif")

    return nlcd

## Export NLCD for all cities

In [7]:
for _, row in cities.iterrows():
    name = row['city']
    aoi  = coords_to_ee_geometry(row['left'], row['bottom'], row['right'], row['top'])
    export_nlcd(name, aoi)


── Processing: brooklyn
   Export task started → Drive/greenspace/brooklyn_NLCD.tif

── Processing: columbus
   Export task started → Drive/greenspace/columbus_NLCD.tif

── Processing: durham
   Export task started → Drive/greenspace/durham_NLCD.tif

── Processing: omaha
   Export task started → Drive/greenspace/omaha_NLCD.tif

── Processing: little_rock
   Export task started → Drive/greenspace/little_rock_NLCD.tif

── Processing: knoxville
   Export task started → Drive/greenspace/knoxville_NLCD.tif

── Processing: cedar_rapids
   Export task started → Drive/greenspace/cedar_rapids_NLCD.tif

── Processing: lynchburg
   Export task started → Drive/greenspace/lynchburg_NLCD.tif

── Processing: richmond
   Export task started → Drive/greenspace/richmond_NLCD.tif

── Processing: san_francisco
   Export task started → Drive/greenspace/san_francisco_NLCD.tif

── Processing: philadelphia
   Export task started → Drive/greenspace/philadelphia_NLCD.tif

── Processing: charlotte
   Export tas